# Phân tích SQL – Dự đoán phí vận chuyển đơn hàng thương mại điện tử Olist
**Nhóm 8 · Lớp AI2107 · Môn ADY201M · FALL26**

---
## 1. Mục tiêu của bước SQL

Sau bước làm sạch, nhóm dùng SQL để **khám phá bảng dữ liệu sạch trước khi xây mô hình**. Bước này trả lời ba câu hỏi:
1. Phí vận chuyển (`freight_value`) có đặc điểm gì?
2. Các yếu tố đã chọn làm đầu vào (trọng lượng, khoảng cách, cùng bang, danh mục) có thật sự ảnh hưởng đến phí không?
3. Dữ liệu có rủi ro nào cần lưu ý trước khi huấn luyện mô hình không?

Kết quả của bước này là cơ sở để chọn biến và chọn cách xử lý ở bước phân tích Python và hồi quy.

## 2. Dữ liệu sử dụng

- **Nguồn:** Brazilian E-Commerce Public Dataset by Olist (Kaggle), giấy phép CC BY-NC-SA 4.0.
- **Bảng dùng trong SQL:** `olist_freight_clean`, là kết quả của bước làm sạch (chỉ giữ đơn đã giao, giai đoạn 01/2017–08/2018, bỏ dòng thiếu tọa độ, cân nặng hoặc kích thước, bỏ phí vận chuyển bằng 0).
- **Đơn vị quan sát:** một sản phẩm trong một đơn hàng, xác định bởi cặp (`order_id`, `order_item_id`).

| Cột | Ý nghĩa |
|---|---|
| `freight_value` | Phí vận chuyển (R$), **biến cần dự đoán** |
| `price` | Giá bán của sản phẩm (R$) |
| `weight_g` | Cân nặng sản phẩm (gam) |
| `length_cm`, `height_cm`, `width_cm`, `volume_cm3` | Kích thước và thể tích sản phẩm |
| `category` | Danh mục sản phẩm (tiếng Anh; `unknown` nếu thiếu) |
| `seller_state`, `customer_state` | Bang của người bán và của khách |
| `same_state` | 1 nếu người bán và khách cùng bang, 0 nếu khác bang |
| `distance_km` | Khoảng cách giữa người bán và khách (km), tính theo công thức Haversine từ tọa độ mã bưu chính |
| `order_purchase_timestamp`, `month`, `t_days` | Thời điểm đặt hàng, tháng đặt hàng, số ngày tính từ 01/01/2017 |
| `order_n_items` | Số sản phẩm trong đơn |

## 3. Chuẩn bị môi trường
Nạp file dữ liệu sạch vào SQLite (cơ sở dữ liệu chạy trong bộ nhớ) và khai báo hàm `sql()` để chạy câu lệnh.

In [ ]:
import sqlite3, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/olist_freight_clean_v3.csv'   # <-- sửa đúng đường dẫn
df = pd.read_csv(CSV_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('olist_freight_clean', conn, index=False, if_exists='replace')

def sql(query):
    """Chạy một câu SQL và trả về bảng kết quả"""
    return pd.read_sql_query(query, conn)

print(df.shape)

### Kiểm tra dữ liệu đã nạp
Xác nhận bảng nạp vào đúng với kết quả của bước làm sạch.

In [ ]:
print('Số dòng, số cột        :', df.shape)
print('Tổng ô thiếu           :', df.isna().sum().sum())
print('Số dòng trùng khóa      :', df.duplicated(['order_id', 'order_item_id']).sum())
print('Giai đoạn đặt hàng      :', df['month'].min(), 'đến', df['month'].max())

Số dòng, số cột        : (108937, 27)
Tổng ô thiếu           : 0
Số dòng trùng khóa      : 0
Giai đoạn đặt hàng      : 2017-01 đến 2018-08


## 4. Tám truy vấn

| Nhóm | Câu | Nội dung |
|---|---|---|
| A. Hiểu dữ liệu | 1 | Tổng quan dữ liệu và phí vận chuyển |
| B. Yếu tố ảnh hưởng đến phí | 2 | Theo trọng lượng |
| B | 3 | Theo khoảng cách |
| B | 4 | Cùng bang so với khác bang |
| B | 5 | Theo danh mục |
| B | 6 | Khoảng cách và trọng lượng cùng tác động |
| C. Kiểm tra rủi ro | 7 | Phí có bị lặp trong đơn nhiều sản phẩm không |
| C | 8 | 10 đơn có phí cao nhất |

---
## A. Hiểu dữ liệu

### Câu 1. Tổng quan dữ liệu và phí vận chuyển
**Câu hỏi:** Dữ liệu có bao nhiêu dòng, bao nhiêu đơn hàng? Phí vận chuyển nhỏ nhất, lớn nhất và trung bình là bao nhiêu?  
**Kỹ thuật SQL:** COUNT, COUNT(DISTINCT), MIN, MAX, AVG

In [ ]:
sql('''
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT order_id) AS so_don_hang,
       MIN(freight_value) AS phi_nho_nhat,
       ROUND(AVG(freight_value), 2) AS phi_trung_binh,
       MAX(freight_value) AS phi_lon_nhat
FROM olist_freight_clean;
''')

so_dong,so_don_hang,phi_nho_nhat,phi_trung_binh,phi_lon_nhat
108937,95377,0.01,20.01,409.68


**Kết quả cho thấy:** Dữ liệu sau làm sạch có 108.937 dòng thuộc 95.377 đơn hàng. Phí trung bình là 20,01 R$ nhưng phí cao nhất lên tới 409,68 R$, gấp khoảng 20 lần mức trung bình.

**Điều rút ra cho mô hình:** Phí ship lệch phải mạnh, nên cần để ý các giá trị rất lớn khi huấn luyện mô hình.

---
## B. Yếu tố ảnh hưởng đến phí

### Câu 2. Phí vận chuyển theo trọng lượng
**Câu hỏi:** Hàng càng nặng thì phí vận chuyển có tăng không?  
**Kỹ thuật SQL:** WITH ... AS (CTE), CASE WHEN, GROUP BY

In [ ]:
sql('''
WITH b AS (
    SELECT freight_value,
           CASE WHEN weight_g < 500 THEN '1. Rat nhe (<500 g)'
               WHEN weight_g < 2000 THEN '2. Nhe (0.5-2 kg)'
               WHEN weight_g < 5000 THEN '3. Trung binh (2-5 kg)'
               ELSE '4. Nang (>5 kg)' END AS nhom_trong_luong
    FROM olist_freight_clean
)
SELECT nhom_trong_luong,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_trung_binh
FROM b
GROUP BY nhom_trong_luong
ORDER BY nhom_trong_luong;
''')

nhom_trong_luong,so_dong,phi_trung_binh
1. Rat nhe (<500 g),43532,15.17
2. Nhe (0.5-2 kg),40632,17.89
3. Trung binh (2-5 kg),11220,22.81
4. Nang (>5 kg),13553,39.62


**Kết quả cho thấy:** Hàng dưới 500 g có phí trung bình 15,17 R$, hàng trên 5 kg là 39,62 R$, gấp khoảng 2,6 lần.

**Điều rút ra cho mô hình:** Trọng lượng là đầu vào có ảnh hưởng rõ, phí tăng dần theo từng nhóm.

### Câu 3. Phí vận chuyển theo khoảng cách
**Câu hỏi:** Khoảng cách giữa người bán và người mua càng xa thì phí có tăng không?  
**Kỹ thuật SQL:** WITH ... AS (CTE), CASE WHEN, GROUP BY

In [ ]:
sql('''
WITH b AS (
    SELECT freight_value,
           CASE WHEN distance_km < 200 THEN '1. Gan (<200 km)'
               WHEN distance_km < 800 THEN '2. Trung binh (200-800 km)'
               WHEN distance_km < 1500 THEN '3. Xa (800-1500 km)'
               ELSE '4. Rat xa (>1500 km)' 
               END AS nhom_khoang_cach
    FROM olist_freight_clean
)
SELECT nhom_khoang_cach,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_trung_binh
FROM b
GROUP BY nhom_khoang_cach
ORDER BY nhom_khoang_cach;
''')

nhom_khoang_cach,so_dong,phi_trung_binh
1. Gan (<200 km),28096,12.66
2. Trung binh (200-800 km),54032,19.80
3. Xa (800-1500 km),17099,23.74
4. Rat xa (>1500 km),9710,35.91


**Kết quả cho thấy:** Đơn giao gần (dưới 200 km) có phí 12,66 R$, đơn giao rất xa (trên 1.500 km) là 35,91 R$, gấp khoảng 2,8 lần.

**Điều rút ra cho mô hình:** Khoảng cách là đầu vào có ảnh hưởng rõ, và cũng tăng dần theo từng nhóm.

### Câu 4. Cùng bang so với khác bang
**Câu hỏi:** Người bán và người mua ở cùng một bang thì phí rẻ hơn bao nhiêu?  
**Kỹ thuật SQL:** GROUP BY

In [ ]:
sql('''
SELECT same_state,
       COUNT(*) AS so_dong,
       ROUND(AVG(distance_km), 0) AS khoang_cach_trung_binh,
       ROUND(AVG(freight_value), 2) AS phi_trung_binh
FROM olist_freight_clean
GROUP BY same_state;
''')

same_state,so_dong,khoang_cach_trung_binh,phi_trung_binh
0,69330,848.0,23.74
1,39607,154.0,13.48


**Kết quả cho thấy:** Cùng bang: phí 13,48 R$ (khoảng cách trung bình 154 km). Khác bang: 23,74 R$ (848 km).

**Điều rút ra cho mô hình:** `same_state` có ý nghĩa, nhưng nó liên quan chặt với khoảng cách nên khi giải thích mô hình cần nói rõ hai biến này cùng phản ánh một ý.

### Câu 5. Phí vận chuyển theo danh mục sản phẩm
**Câu hỏi:** Danh mục nào có phí vận chuyển cao nhất (chỉ xét danh mục có từ 300 dòng trở lên)?  
**Kỹ thuật SQL:** GROUP BY, HAVING, ORDER BY

In [ ]:
sql('''
SELECT category,
       COUNT(*) AS so_dong,
       ROUND(AVG(weight_g), 0) AS trong_luong_trung_binh_g,
       ROUND(AVG(freight_value), 2) AS phi_trung_binh
FROM olist_freight_clean
GROUP BY category
HAVING COUNT(*) >= 300
ORDER BY phi_trung_binh DESC
LIMIT 10;
''')

category,so_dong,trong_luong_trung_binh_g,phi_trung_binh
office_furniture,1655,11329.0,40.17
furniture_living_room,443,8727.0,37.43
luggage_accessories,1073,5788.0,27.83
musical_instruments,650,3135.0,27.46
construction_tools_lights,300,3358.0,24.80
small_appliances,658,2929.0,23.33
garden_tools,4196,2827.0,22.94
home_construction,596,3037.0,22.84
baby,2961,3271.0,22.24
construction_tools_construction,909,3080.0,22.08


**Kết quả cho thấy:** Đắt nhất là đồ nội thất văn phòng (40,17 R$) và nội thất phòng khách (37,43 R$), đồng thời cũng là hai nhóm nặng nhất (khoảng 11,3 kg và 8,7 kg).

**Điều rút ra cho mô hình:** Danh mục đắt phần lớn vì hàng nặng chứ không phải vì bản thân danh mục. Cần giữ trọng lượng trong mô hình để không hiểu sai vai trò của danh mục.

### Câu 6. Khoảng cách và trọng lượng cùng tác động
**Câu hỏi:** Khi xét đồng thời khoảng cách và trọng lượng, phí thay đổi thế nào?  
**Kỹ thuật SQL:** WITH ... AS (CTE), CASE WHEN bên trong AVG (bảng chéo)

In [ ]:
sql('''
WITH b AS (
    SELECT freight_value, weight_g,
           CASE WHEN distance_km < 200 THEN '1. Gan (<200 km)'
               WHEN distance_km < 800 THEN '2. Trung binh (200-800 km)'
               WHEN distance_km < 1500 THEN '3. Xa (800-1500 km)'
               ELSE '4. Rat xa (>1500 km)' END AS nhom_khoang_cach
    FROM olist_freight_clean
)
SELECT nhom_khoang_cach,
       ROUND(AVG(CASE WHEN weight_g < 2000 THEN freight_value END), 2) AS hang_nhe_duoi_2kg,
       ROUND(AVG(CASE WHEN weight_g >= 2000 AND weight_g < 5000 THEN freight_value END), 2) AS hang_2_den_5kg,
       ROUND(AVG(CASE WHEN weight_g >= 5000 THEN freight_value END), 2) AS hang_tren_5kg
FROM b
GROUP BY nhom_khoang_cach
ORDER BY nhom_khoang_cach;
''')

nhom_khoang_cach,hang_nhe_duoi_2kg,hang_2_den_5kg,hang_tren_5kg
1. Gan (<200 km),10.26,15.14,25.93
2. Trung binh (200-800 km),16.30,22.04,38.92
3. Xa (800-1500 km),19.57,26.86,48.10
4. Rat xa (>1500 km),30.15,39.94,70.47


**Kết quả cho thấy:** Hàng nhẹ ở gần chỉ 10,26 R$, hàng trên 5 kg ở rất xa lên tới 70,47 R$. Mỗi dòng và mỗi cột đều tăng dần.

**Điều rút ra cho mô hình:** Hai yếu tố cộng hưởng với nhau: khoảng cách tăng thì hàng nặng tăng phí mạnh hơn hàng nhẹ. Đây là cơ sở để thử thêm mô hình phi tuyến như Random Forest bên cạnh hồi quy tuyến tính.

---
## C. Kiểm tra rủi ro của dữ liệu

### Câu 7. Đơn nhiều sản phẩm: phí có bị lặp lại không?
**Câu hỏi:** Với đơn có từ 2 sản phẩm trở lên, có bao nhiêu đơn có phí vận chuyển giống hệt nhau ở mọi sản phẩm?  
**Kỹ thuật SQL:** WITH ... AS (CTE), GROUP BY, HAVING

In [ ]:
sql('''
WITH don_nhieu_sp AS (
    SELECT order_id,
           MIN(freight_value) AS phi_nho_nhat,
           MAX(freight_value) AS phi_lon_nhat
    FROM olist_freight_clean
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS so_don_nhieu_san_pham,
       SUM(CASE WHEN phi_nho_nhat = phi_lon_nhat THEN 1 ELSE 0 END) AS so_don_phi_giong_nhau,
       ROUND(100.0 * SUM(CASE WHEN phi_nho_nhat = phi_lon_nhat THEN 1 ELSE 0 END) / COUNT(*), 1) AS ty_le_phan_tram
FROM don_nhieu_sp;
''')

so_don_nhieu_san_pham,so_don_phi_giong_nhau,ty_le_phan_tram
9516,7526,79.1


**Kết quả cho thấy:** Trong 9.516 đơn có từ 2 sản phẩm trở lên, có 7.526 đơn (79,1%) mà mọi sản phẩm đều mang cùng một mức phí.

**Điều rút ra cho mô hình:** Nhiều khả năng phí là của cả đơn và bị lặp lại cho từng sản phẩm. Đây là hạn chế cần nêu rõ trong báo cáo và cân nhắc khi chọn đơn vị dự đoán.

### Câu 8. 10 đơn có phí vận chuyển cao nhất
**Câu hỏi:** Những đơn có phí cao bất thường là đơn như thế nào? Có phải lỗi dữ liệu không?  
**Kỹ thuật SQL:** WHERE, ORDER BY, TOP/LIMIT

In [ ]:
sql('''
SELECT order_id, category, weight_g, ROUND(distance_km, 0) AS khoang_cach_km, freight_value
FROM olist_freight_clean
ORDER BY freight_value DESC
LIMIT 10;
''')

order_id,category,weight_g,khoang_cach_km,freight_value
a77e1550db865202c56b19ddc6dc4d53,baby,14675.0,1865.0,409.68
076d1555fb53a89b0ef4d529e527a0f6,housewares,30000.0,457.0,375.28
3fde74c28a3d5d618c00f26d51baafa0,housewares,30000.0,450.0,375.28
9f49bd16053df810384e793386312674,housewares,25250.0,617.0,339.59
264a7e199467906c0727394df82d1a6a,health_beauty,30000.0,1708.0,338.30
c7a07ddd52bbe18b61da49a8d89853d3,health_beauty,30000.0,433.0,322.10
0b6230647ed16f4b3e70282dc4b5b87f,health_beauty,30000.0,632.0,321.88
0822bcde10bb5d023755a71bc8f7797f,construction_tools_construction,30000.0,242.0,321.46
43bdbd9dc0931d72befdf4765af6c442,industry_commerce_and_business,11300.0,2820.0,317.47
6ddfbf514959b49b6410c01ad93054bb,construction_tools_construction,30000.0,1689.0,314.40


**Kết quả cho thấy:** Cả 10 đơn đều là hàng nặng từ 11,3 kg đến 30 kg, phí từ 314 đến 410 R$.

**Điều rút ra cho mô hình:** Phí cao do hàng nặng thật, không phải lỗi dữ liệu, nên giữ lại những dòng này.

---
## 5. Tổng hợp phát hiện

| Câu | Phát hiện | Quyết định cho bước sau |
|---|---|---|
| 1 | Phí trung bình 20,01 R$, cao nhất 409,68 R$ (gấp khoảng 20 lần) | Phí lệch phải, cân nhắc log-transform khi mô hình |
| 2 | Phí tăng từ 15,17 lên 39,62 R$ theo trọng lượng | Giữ `weight_g` làm đầu vào |
| 3 | Phí tăng từ 12,66 lên 35,91 R$ theo khoảng cách | Giữ `distance_km` làm đầu vào |
| 4 | Cùng bang 13,48 R$, khác bang 23,74 R$ | Giữ `same_state`, lưu ý liên quan chặt với khoảng cách |
| 5 | Nội thất văn phòng và phòng khách đắt nhất, cũng là nhóm nặng nhất | Giữ trọng lượng cùng danh mục để tránh hiểu sai vai trò danh mục |
| 6 | Hàng nặng ở xa tăng phí mạnh hơn hàng nhẹ ở xa | Thử thêm Random Forest bên cạnh hồi quy tuyến tính |
| 7 | 79,1% đơn nhiều sản phẩm có phí giống hệt nhau | Nêu rõ là hạn chế, cân nhắc đơn vị dự đoán |
| 8 | 10 đơn phí cao nhất đều là hàng nặng 11,3–30 kg | Giữ lại, không phải lỗi dữ liệu |

## 6. Hạn chế và bước tiếp theo

**Hạn chế:**
- Phí vận chuyển có thể là phí của cả đơn được lặp cho từng sản phẩm (câu 7), nên phí theo từng dòng chưa chắc phản ánh đúng chi phí của riêng sản phẩm đó.
- Khoảng cách tính từ tọa độ mã bưu chính, không phải địa chỉ chi tiết.
- Dữ liệu không có thông tin hãng vận chuyển, giá cước hay giá nhiên liệu.
- Kết quả SQL chỉ mô tả mối quan hệ, chưa phải kết luận về nguyên nhân.

**Bước tiếp theo (Python):** vẽ biểu đồ phân phối và quan hệ giữa phí với từng biến, tính tương quan (gồm cặp trọng lượng – thể tích), rồi huấn luyện và so sánh các mô hình hồi quy bằng R², MAE và cross-validation.